# Breast cancer screening on real mammograms vs the NHS benchmark

**Goal:** train a CNN on *screening* mammograms and compare it fairly with the NHS first reader
(sensitivity 43.7%, specificity 95.2%; Google AI 54.1% / 94.3%, *Nature Cancer*).

**What's different from `model_creation.ipynb`:**
1. **Realistic data**: RSNA Screening Mammography dataset (real screening population, ~2% cancers, patient IDs).
2. **Patient-level split**: no woman appears in more than one of train / val / test, so no leakage.
3. **Held-out test set**: touched once, at the very end.
4. **Augmentation only on training data**, after splitting.
5. **Fair metric**: sensitivity *at the NHS reader's specificity* (95.2%), not accuracy.

**Run it on Kaggle with a GPU** (free). On CPU it will take days.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import kagglehub
from keras import layers, optimizers, losses, callbacks
from keras.applications import MobileNetV3Large
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt

SEED = 123
IMG_SIZE = 512        # mammograms need more resolution than 224; tumours can be tiny
BATCH_SIZE = 16
POS_OVERSAMPLE = 10   # repeat cancer images in TRAINING only to handle ~2% prevalence
NHS_SPECIFICITY = 0.952

keras.utils.set_random_seed(SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Data
Labels come from the RSNA competition (accept its rules on Kaggle first).
The original DICOMs are ~300 GB, so images come from a community PNG conversion.
**Check the PNG dataset slug exists**; any PNG copy named `<image_id>.png` will work.

In [ ]:
CSV_PATH = kagglehub.competition_download("rsna-breast-cancer-detection", path="train.csv")
PNG_DIR = kagglehub.dataset_download("radek1/rsna-mammography-images-as-pngs")  # verify / swap if needed

df = pd.read_csv(CSV_PATH)
df = df[df["view"].isin(["CC", "MLO"])]  # the two standard screening views

all_pngs = glob.glob(os.path.join(PNG_DIR, "**", "*.png"), recursive=True)
preferred = [p for p in all_pngs if str(IMG_SIZE) in os.path.dirname(p)] or all_pngs
path_by_id = {}
for p in preferred:
    stem = os.path.splitext(os.path.basename(p))[0]
    if stem.isdigit():
        path_by_id[int(stem)] = p

df["path"] = df["image_id"].map(path_by_id)
print(f"Matched {df['path'].notna().sum()} / {len(df)} images to PNG files")
df = df.dropna(subset=["path"]).reset_index(drop=True)
print(df["cancer"].value_counts(normalize=True).rename("share of images"))

## 2. Patient-level split (70 / 15 / 15, stratified by whether the woman has cancer)

In [ ]:
patients = df.groupby("patient_id")["cancer"].max().reset_index()
train_p, temp_p = train_test_split(patients, test_size=0.30, stratify=patients["cancer"], random_state=SEED)
val_p, test_p = train_test_split(temp_p, test_size=0.50, stratify=temp_p["cancer"], random_state=SEED)

split_of = {**dict.fromkeys(train_p["patient_id"], "train"),
            **dict.fromkeys(val_p["patient_id"], "val"),
            **dict.fromkeys(test_p["patient_id"], "test")}
df["split"] = df["patient_id"].map(split_of)

# Leakage check: no patient in more than one split
assert df.groupby("patient_id")["split"].nunique().max() == 1

print(df.groupby("split").agg(images=("image_id", "size"),
                              women=("patient_id", "nunique"),
                              cancer_rate=("cancer", "mean")))

## 3. Input pipeline (augmentation on training only)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE)  # keeps aspect ratio
    return img, label  # 0-255 float; MobileNetV3 does its own preprocessing

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["cancer"].values.astype("float32")))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_df = df[df["split"] == "train"]
pos, neg = train_df[train_df["cancer"] == 1], train_df[train_df["cancer"] == 0]
train_bal = pd.concat([neg, pos.sample(len(pos) * POS_OVERSAMPLE, replace=True, random_state=SEED)])
print(f"Training images after oversampling: {len(train_bal)} ({train_bal['cancer'].mean():.1%} cancer)")

train_ds = make_ds(train_bal, training=True)
val_ds = make_ds(df[df["split"] == "val"], training=False)

## 4. Model: MobileNetV3Large transfer learning (same backbone as before)

In [ ]:
base = MobileNetV3Large(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                        weights="imagenet", include_preprocessing=True)
base.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)  # keep BatchNorm in inference mode, even when fine-tuning
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

def compile_model(lr):
    model.compile(optimizer=optimizers.Adam(lr),
                  loss=losses.BinaryCrossentropy(),
                  metrics=[keras.metrics.AUC(name="auc")])

cbs = [callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True),
       callbacks.ModelCheckpoint("mammo_best.keras", monitor="val_auc", mode="max", save_best_only=True)]

# Stage 1: train the new head only
compile_model(1e-3)
model.fit(train_ds, validation_data=val_ds, epochs=3)

# Stage 2: fine-tune the whole network with a small learning rate
base.trainable = True
compile_model(1e-5)
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs)

## 5. Fair comparison with the NHS
Screening decisions are per **woman**, so image scores are combined:
mean over images of each breast, then max over her two breasts.
The threshold is chosen on **validation** to match the NHS reader's 95.2% specificity, then applied once to **test**.

In [ ]:
def woman_scores(frame):
    out = frame[["patient_id", "laterality", "cancer"]].copy()
    out["prob"] = model.predict(make_ds(frame, training=False), verbose=0).ravel()
    breast = out.groupby(["patient_id", "laterality"]).agg(cancer=("cancer", "max"), prob=("prob", "mean"))
    return breast.groupby("patient_id").agg(cancer=("cancer", "max"), prob=("prob", "max"))

val_w = woman_scores(df[df["split"] == "val"])
test_w = woman_scores(df[df["split"] == "test"])

# Threshold = score that 95.2% of healthy validation women fall below
threshold = np.quantile(val_w.loc[val_w["cancer"] == 0, "prob"], NHS_SPECIFICITY)

pred = (test_w["prob"] >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(test_w["cancer"], pred, labels=[0, 1]).ravel()
sens, spec = tp / (tp + fn), tn / (tn + fp)
auc = roc_auc_score(test_w["cancer"], test_w["prob"])

results = pd.DataFrame({
    "Sensitivity": [0.437, 0.541, sens],
    "Specificity": [0.952, 0.943, spec],
}, index=["NHS first reader", "Google AI (Nature Cancer)", "My model (test set)"])
print((results * 100).round(1).astype(str) + "%")
print(f"\nTest AUC: {auc:.3f} | women: {len(test_w)} | cancers: {int(test_w['cancer'].sum())} | TP={tp} FN={fn} FP={fp}")

In [ ]:
fpr, tpr, _ = roc_curve(test_w["cancer"], test_w["prob"])
plt.figure(figsize=(6, 6))
plt.plot(1 - fpr, tpr, label=f"My model (AUC {auc:.3f})")
plt.scatter([0.952], [0.437], marker="x", s=100, label="NHS first reader")
plt.scatter([0.943], [0.541], marker="o", s=80, label="Google AI")
plt.xlim(0.8, 1.0); plt.ylim(0, 1)
plt.xlabel("Specificity"); plt.ylabel("Sensitivity")
plt.title("Per-woman ROC on held-out test set"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 6. Read before claiming anything
- **Still not a perfect like-for-like.** The NHS figures count cancers found up to ~3 years later (interval cancers).
  RSNA labels mostly count cancers confirmed at that screen, so your sensitivity will look *higher* than a true equivalent.
- **Different population and hardware.** RSNA images come from the US and Australia, not NHS services.
- **Cancers per 1,000 women isn't comparable**, because RSNA has a higher cancer rate than the NHS screening population.
- **Fair wording:** *"At the NHS first reader's specificity, my model reached X% sensitivity on a held-out, patient-level test set."*

In [ ]:
model.save("mammo_final.keras")